# Initial-land restart processing

This notebook is the interactive workflow driver for ELM restart aggregation and optional conservative regridding. It calls reusable functions from util/ directly; optional command-line entry points remain under script/initial_land/ for unattended runs. Existing products are reused unless FORCE_COMPUTE = True. Both execution switches default to False; configure and inspect the paths first.


In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "analysis_lnd_init"

In [ ]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)

## Workflow controls

- `VARIABLES`: any numeric restart variable containing a `column` dimension. Common choices include `H2OSOI`, `TSA`, and `SOILM`.
- `SUM_VARIABLES`: selected extensive variables that must be summed across columns. Other selected variables use area-weighted means.
- `RUN_AGGREGATION`: build or reuse the column-to-gridcell NetCDF product.
- `RUN_REGRID`: build or reuse the source SCRIP grid, conservative map, and regridded restart.
- `CORNER_PERMUTATION`: use `keep` normally; `0132` and `0321` are available for source domains whose corner order is rejected by ESMF.
- `FORCE_COMPUTE`: replace all applicable cached products only when set to `True`.

All generated files are written beneath `DIAGNOSTIC_OUTPUT_ROOT/data/analysis_lnd_init/restart/`.


In [ ]:
from configs.initial_land_experiment_config import DEFAULT_RESTART_REGRID
from util.initial_lnd_restart_aggregation import ColumnToGridcellAggregator
from util.initial_lnd_restart_regridding import conservative_regrid_restart
import xarray as xr

# =============================================================================
# USER SETUP
# =============================================================================
RESTART_FILE = Path("/path/to/ELM_restart.nc")
SOURCE_DOMAIN_FILE = Path("/path/to/land_domain.nc")
RESTART_OUTPUT_DIR = WORKFLOW_DATA_DIR / "restart"
RESTART_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REGRID_CONFIG = dict(DEFAULT_RESTART_REGRID)
REGRID_CONFIG.update({
    "out_agg": RESTART_OUTPUT_DIR / "agg_gridcell.nc",
    "dst_scrip": INPUT_DATA_ROOT / "reference/regrid_maps/cmip6_180x360_scrip.20181001.nc",
    "src_scrip": RESTART_OUTPUT_DIR / "src_unstruct_from_agg.scrip.nc",
    "map_file": RESTART_OUTPUT_DIR / "map_unstruct_to_180x360_aave.nc",
    "regrid_out": RESTART_OUTPUT_DIR / "elm_180x360_aave.nc",
})
AGGREGATED_FILE = Path(REGRID_CONFIG["out_agg"])
VARIABLES = ["H2OSOI", "TSA", "SOILM"]
SUM_VARIABLES = []
CORNER_PERMUTATION = "keep"  # alternatives: "0132", "0321"
NCREMAP_DEBUG_LEVEL = 1
RUN_AGGREGATION = False
RUN_REGRID = False
FORCE_COMPUTE = False

if RUN_AGGREGATION:
    if AGGREGATED_FILE.is_file() and not FORCE_COMPUTE:
        print(f"Reusing aggregated restart: {AGGREGATED_FILE}")
        with xr.open_dataset(AGGREGATED_FILE) as cached:
            aggregated = cached.load()
    else:
        for path in (RESTART_FILE, SOURCE_DOMAIN_FILE):
            if not path.is_file():
                raise FileNotFoundError(path)
        with ColumnToGridcellAggregator(RESTART_FILE, SOURCE_DOMAIN_FILE) as aggregator:
            aggregated = aggregator.export(
                AGGREGATED_FILE,
                varnames=VARIABLES,
                sum_vars=SUM_VARIABLES,
                compress=True,
                overwrite=FORCE_COMPUTE,
            )
    display(aggregated)

if RUN_REGRID:
    if not AGGREGATED_FILE.is_file():
        raise FileNotFoundError(
            f"Aggregated restart not found: {AGGREGATED_FILE}. "
            "Run aggregation first or point REGRID_CONFIG['out_agg'] to an existing cache."
        )
    regridded_file = conservative_regrid_restart(
        AGGREGATED_FILE,
        REGRID_CONFIG["dst_scrip"],
        REGRID_CONFIG["src_scrip"],
        REGRID_CONFIG["map_file"],
        REGRID_CONFIG["regrid_out"],
        corner_permutation=CORNER_PERMUTATION,
        variables=VARIABLES,
        force=FORCE_COMPUTE,
        debug_level=NCREMAP_DEBUG_LEVEL,
    )
    print(f"Regridded restart: {regridded_file}")
    with xr.open_dataset(regridded_file) as regridded:
        display(regridded)
